# Olist Marketplace Operations & Customer Experience Intelligence
## Data Preparation and Investigation Dataset Engineering

This notebook prepares the Olist Brazilian e-commerce dataset for downstream SQL analysis and dashboard development.

### Objectives
- Load and inventory the raw datasets
- Perform data-quality checks and cleaning
- Build a one-row-per-order investigation dataset
- Engineer operational and exception indicators
- Create customer- and seller-level investigation datasets
- Export final processed datasets for SQL and dashboard analysis

> **Note:** The investigation scoring framework identifies cases for further operational review. It does not claim to detect fraud, AML, or other regulated financial crime activity.


## 1. Setup and Data Loading

In [2]:
import os
import pandas as pd

DATA_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)


In [3]:
customers = pd.read_csv(f"{DATA_PATH}/olist_customers_dataset.csv")
geolocation = pd.read_csv(f"{DATA_PATH}/olist_geolocation_dataset.csv")
order_items = pd.read_csv(f"{DATA_PATH}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{DATA_PATH}/olist_order_payments_dataset.csv")
reviews = pd.read_csv(f"{DATA_PATH}/olist_order_reviews_dataset.csv")
orders = pd.read_csv(f"{DATA_PATH}/olist_orders_dataset.csv")
products = pd.read_csv(f"{DATA_PATH}/olist_products_dataset.csv")
sellers = pd.read_csv(f"{DATA_PATH}/olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    f"{DATA_PATH}/product_category_name_translation.csv"
)

datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

print("Datasets loaded:", len(datasets))


Datasets loaded: 9


## 2. Data Inventory and Quality Assessment

In [4]:
data_quality_inventory = pd.DataFrame({
    "Dataset": list(datasets.keys()),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()],
    "Missing Values": [df.isnull().sum().sum() for df in datasets.values()],
    "Duplicate Rows": [df.duplicated().sum() for df in datasets.values()]
})

data_quality_inventory


,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,customers,99441,5,0,0
1,geolocation,1000163,5,0,261831
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,2448,0
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


In [5]:
def dataset_profile(df):
    return pd.DataFrame({
        "Data Type": df.dtypes.astype(str),
        "Missing Values": df.isnull().sum(),
        "Missing %": (df.isnull().mean() * 100).round(2),
        "Unique Values": df.nunique()
    })

profiles = {
    name: dataset_profile(df)
    for name, df in datasets.items()
}

for name, profile in profiles.items():
    print(f"\n{name.upper()}")
    display(profile)



CUSTOMERS


,Data Type,Missing Values,Missing %,Unique Values
customer_id,object,0,0.0,99441
customer_unique_id,object,0,0.0,96096
customer_zip_code_prefix,int64,0,0.0,14994
customer_city,object,0,0.0,4119
customer_state,object,0,0.0,27



GEOLOCATION


,Data Type,Missing Values,Missing %,Unique Values
geolocation_zip_code_prefix,int64,0,0.0,19015
geolocation_lat,float64,0,0.0,717360
geolocation_lng,float64,0,0.0,717613
geolocation_city,object,0,0.0,8011
geolocation_state,object,0,0.0,27



ORDER_ITEMS


...

,Data Type,Missing Values,Missing %,Unique Values
product_id,object,0,0.00,32951
product_category_name,object,610,1.85,73
product_name_lenght,float64,610,1.85,66
product_description_lenght,float64,610,1.85,2960
product_photos_qty,float64,610,1.85,19
product_weight_g,float64,2,0.01,2204
product_length_cm,float64,2,0.01,99
product_height_cm,float64,2,0.01,102
product_width_cm,float64,2,0.01,95



SELLERS


,Data Type,Missing Values,Missing %,Unique Values
seller_id,object,0,0.0,3095
seller_zip_code_prefix,int64,0,0.0,2246
seller_city,object,0,0.0,611
seller_state,object,0,0.0,23



CATEGORY_TRANSLATION


,Data Type,Missing Values,Missing %,Unique Values
product_category_name,object,0,0.0,71
product_category_name_english,object,0,0.0,71


## 3. Data Cleaning

Cleaning decisions:
- Raw datasets remain unchanged.
- Exact duplicate geolocation records are removed.
- Missing product categories and descriptive fields are retained without dropping products.
- Operational timestamps are converted to datetime.
- Missing review comments and missing delivery timestamps are preserved because they are meaningful in the source data.


In [6]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()


In [7]:
geolocation_clean = (
    geolocation_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Rows after removing duplicate geolocation records:", geolocation_clean.shape[0])


Rows after removing duplicate geolocation records: 738332


In [8]:
products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

products_clean["product_name_lenght"] = (
    products_clean["product_name_lenght"]
    .fillna(0)
)

products_clean["product_description_lenght"] = (
    products_clean["product_description_lenght"]
    .fillna(0)
)

products_clean["product_photos_qty"] = (
    products_clean["product_photos_qty"]
    .fillna(0)
)


In [9]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_columns:
    reviews_clean[col] = pd.to_datetime(
        reviews_clean[col],
        errors="coerce"
    )


In [10]:
cleaned_datasets = {
    "customers": customers_clean,
    "geolocation": geolocation_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "orders": orders_clean,
    "products": products_clean,
    "sellers": sellers_clean,
    "category_translation": category_translation_clean
}

cleaning_summary = pd.DataFrame({
    "Dataset": list(cleaned_datasets.keys()),
    "Rows": [df.shape[0] for df in cleaned_datasets.values()],
    "Columns": [df.shape[1] for df in cleaned_datasets.values()],
    "Missing Values": [
        df.isnull().sum().sum()
        for df in cleaned_datasets.values()
    ],
    "Duplicate Rows": [
        df.duplicated().sum()
        for df in cleaned_datasets.values()
    ]
})

cleaning_summary


,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,customers,99441,5,0,0
1,geolocation,738332,5,0,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,8,0
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


## 4. Order-Level Dataset Engineering

### 4.1 Aggregate payment records to one row per order

This prevents row multiplication when payment data is joined to the order-level dataset.


In [11]:
payment_summary = (
    payments_clean
    .groupby("order_id")
    .agg(
        payment_records=("payment_sequential", "count"),
        payment_types=("payment_type", "nunique"),
        total_payment_value=("payment_value", "sum"),
        total_installments=("payment_installments", "sum"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

print("Payment Summary:", payment_summary.shape)


Payment Summary: (99440, 6)


### 4.2 Aggregate order items to one row per order

In [12]:
order_item_summary = (
    order_items_clean
    .groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        total_product_value=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
    .reset_index()
)

print("Order Item Summary:", order_item_summary.shape)


Order Item Summary: (98666, 6)


### 4.3 Aggregate review records to one row per order

In [13]:
review_summary = (
    reviews_clean
    .groupby("order_id")
    .agg(
        review_records=("review_id", "count"),
        average_review_score=("review_score", "mean"),
        minimum_review_score=("review_score", "min"),
        maximum_review_score=("review_score", "max")
    )
    .reset_index()
)

print("Review Summary:", review_summary.shape)


Review Summary: (98673, 5)


### 4.4 Build the master order-level dataset

In [14]:
master_orders = (
    orders_clean
    .merge(
        customers_clean,
        on="customer_id",
        how="left"
    )
    .merge(
        payment_summary,
        on="order_id",
        how="left"
    )
    .merge(
        order_item_summary,
        on="order_id",
        how="left"
    )
    .merge(
        review_summary,
        on="order_id",
        how="left"
    )
)

print("Master Dataset Shape:", master_orders.shape)
print("Unique Orders:", master_orders["order_id"].nunique())


Master Dataset Shape: (99441, 26)
Unique Orders: 99441


## 5. Operational Metrics and Investigation Signals

In [15]:
master_orders["delivery_days"] = (
    master_orders["order_delivered_customer_date"]
    - master_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

master_orders["approval_hours"] = (
    master_orders["order_approved_at"]
    - master_orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

master_orders["carrier_handoff_days"] = (
    master_orders["order_delivered_carrier_date"]
    - master_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Positive variance indicates delivery after the estimated delivery date.
master_orders["delivery_variance_days"] = (
    master_orders["order_delivered_customer_date"]
    - master_orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

master_orders["late_delivery_flag"] = (
    master_orders["delivery_variance_days"] > 0
).astype(int)


In [16]:
master_orders["multiple_payment_flag"] = (
    master_orders["payment_records"] > 1
).astype(int)

master_orders["multiple_seller_flag"] = (
    master_orders["unique_sellers"] > 1
).astype(int)

master_orders["low_review_flag"] = (
    master_orders["average_review_score"] <= 2
).astype(int)


### 5.1 Customer history metrics

In [17]:
customer_order_metrics = (
    master_orders
    .groupby("customer_unique_id")
    .agg(
        customer_total_orders=("order_id", "count"),
        customer_total_spend=("total_payment_value", "sum"),
        customer_average_order_value=("total_payment_value", "mean")
    )
    .reset_index()
)

master_orders = master_orders.merge(
    customer_order_metrics,
    on="customer_unique_id",
    how="left"
)

master_orders["repeat_customer_flag"] = (
    master_orders["customer_total_orders"] > 1
).astype(int)


## 6. Investigation Priority Model

The model combines operational and transaction-related exception signals to prioritize orders for further review. Thresholds are derived from the dataset distribution rather than using arbitrary fixed values.


In [18]:
high_value_threshold = master_orders["total_payment_value"].quantile(0.90)

master_orders["high_value_flag"] = (
    master_orders["total_payment_value"] >= high_value_threshold
).astype(int)

print("High-value threshold:", round(high_value_threshold, 2))


High-value threshold: 308.24


In [19]:
late_deliveries = master_orders.loc[
    master_orders["delivery_variance_days"] > 0,
    "delivery_variance_days"
]

severe_delay_threshold = late_deliveries.quantile(0.90)

master_orders["severe_delay_flag"] = (
    master_orders["delivery_variance_days"] >= severe_delay_threshold
).astype(int)

print(
    "Severe delay threshold:",
    round(severe_delay_threshold, 2),
    "days"
)


Severe delay threshold: 21.53 days


In [20]:
master_orders["investigation_score"] = (
    master_orders["high_value_flag"]
    + master_orders["multiple_payment_flag"]
    + master_orders["multiple_seller_flag"]
    + master_orders["late_delivery_flag"]
    + master_orders["severe_delay_flag"]
    + master_orders["low_review_flag"]
)

def assign_priority(score):
    if score >= 4:
        return "High"
    elif score >= 2:
        return "Medium"
    return "Low"

master_orders["investigation_priority"] = (
    master_orders["investigation_score"]
    .apply(assign_priority)
)


In [21]:
case_summary = (
    master_orders["investigation_priority"]
    .value_counts()
    .rename_axis("Priority")
    .reset_index(name="Cases")
)

case_summary


,Priority,Cases
0,Low,92128
1,Medium,7216
2,High,97


## 7. Seller Investigation Dataset

In [22]:
seller_order_analysis = (
    order_items_clean[
        ["order_id", "seller_id", "product_id", "price", "freight_value"]
    ]
    .merge(
        master_orders[
            [
                "order_id",
                "order_status",
                "delivery_days",
                "delivery_variance_days",
                "late_delivery_flag",
                "severe_delay_flag",
                "average_review_score",
                "low_review_flag",
                "investigation_score",
                "investigation_priority"
            ]
        ],
        on="order_id",
        how="left"
    )
)

seller_investigation = (
    seller_order_analysis
    .groupby("seller_id")
    .agg(
        orders_handled=("order_id", "nunique"),
        total_product_value=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delivery_variance_days=("delivery_variance_days", "mean"),
        late_orders=("late_delivery_flag", "sum"),
        severe_delay_orders=("severe_delay_flag", "sum"),
        avg_review_score=("average_review_score", "mean"),
        low_review_orders=("low_review_flag", "sum"),
        avg_investigation_score=("investigation_score", "mean")
    )
    .reset_index()
)

seller_investigation["late_delivery_rate"] = (
    seller_investigation["late_orders"]
    / seller_investigation["orders_handled"]
)

seller_investigation["severe_delay_rate"] = (
    seller_investigation["severe_delay_orders"]
    / seller_investigation["orders_handled"]
)

seller_investigation["low_review_rate"] = (
    seller_investigation["low_review_orders"]
    / seller_investigation["orders_handled"]
)

seller_investigation = seller_investigation.merge(
    sellers_clean[
        ["seller_id", "seller_city", "seller_state"]
    ],
    on="seller_id",
    how="left"
)

print("Seller Investigation Shape:", seller_investigation.shape)
print("Unique Sellers:", seller_investigation["seller_id"].nunique())


Seller Investigation Shape: (3095, 16)
Unique Sellers: 3095


## 8. Customer Investigation Dataset

In [23]:
customer_investigation = (
    master_orders
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "count"),
        total_spend=("total_payment_value", "sum"),
        avg_order_value=("total_payment_value", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        late_orders=("late_delivery_flag", "sum"),
        severe_delay_orders=("severe_delay_flag", "sum"),
        avg_review_score=("average_review_score", "mean"),
        low_review_orders=("low_review_flag", "sum"),
        multiple_payment_orders=("multiple_payment_flag", "sum"),
        high_value_orders=("high_value_flag", "sum"),
        avg_investigation_score=("investigation_score", "mean"),
        high_priority_cases=(
            "investigation_priority",
            lambda x: (x == "High").sum()
        )
    )
    .reset_index()
)

customer_investigation["late_order_rate"] = (
    customer_investigation["late_orders"]
    / customer_investigation["total_orders"]
)

customer_investigation["low_review_rate"] = (
    customer_investigation["low_review_orders"]
    / customer_investigation["total_orders"]
)

customer_investigation["multiple_payment_rate"] = (
    customer_investigation["multiple_payment_orders"]
    / customer_investigation["total_orders"]
)

customer_investigation["repeat_customer_flag"] = (
    customer_investigation["total_orders"] > 1
).astype(int)


In [24]:
customer_locations = (
    master_orders
    .sort_values("order_purchase_timestamp")
    .drop_duplicates(
        subset="customer_unique_id",
        keep="last"
    )
    [
        [
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ]
)

customer_investigation = customer_investigation.merge(
    customer_locations,
    on="customer_unique_id",
    how="left"
)

print("Customer Investigation Shape:", customer_investigation.shape)
print(
    "Unique Customers:",
    customer_investigation["customer_unique_id"].nunique()
)


Customer Investigation Shape: (96096, 19)
Unique Customers: 96096


## 9. Export Final Processed Datasets

In [25]:
master_orders.to_csv(
    f"{PROCESSED_PATH}/olist_order_investigation.csv",
    index=False
)

seller_investigation.to_csv(
    f"{PROCESSED_PATH}/olist_seller_investigation.csv",
    index=False
)

customer_investigation.to_csv(
    f"{PROCESSED_PATH}/olist_customer_investigation.csv",
    index=False
)

products_clean.to_csv(
    f"{PROCESSED_PATH}/olist_products_clean.csv",
    index=False
)

order_items_clean.to_csv(
    f"{PROCESSED_PATH}/olist_order_items_clean.csv",
    index=False
)

print("Processed files saved:")
for file_name in sorted(os.listdir(PROCESSED_PATH)):
    print("-", file_name)



Processed files saved:
- olist_customer_investigation.csv
- olist_order_investigation.csv
- olist_order_items_clean.csv
- olist_products_clean.csv
- olist_seller_investigation.csv


## Final Outputs

This notebook produces five processed datasets that are used for SQL analysis and dashboard development:

1. **`olist_order_investigation.csv`**  
   Order-level dataset containing operational performance, delivery metrics, payment information, customer review metrics, and investigation indicators.

2. **`olist_seller_investigation.csv`**  
   Seller-level dataset containing operational performance, delivery metrics, customer experience metrics, and exception indicators.

3. **`olist_customer_investigation.csv`**  
   Customer-level dataset containing customer behaviour, spending patterns, order history, delivery issues, and investigation metrics.

4. **`olist_products_clean.csv`**  
   Cleaned product reference dataset containing product category and product characteristic information.

5. **`olist_order_items_clean.csv`**  
   Cleaned order-item level transaction dataset linking orders, products, and sellers, including product value and freight value.

These processed datasets form the foundation for the next stages of the project:

- SQL Analysis
- Business Investigation
- Dashboard Development
- Insight Generation
